In [0]:

# Healthcare Data Engineering - Bronze Layer
# Step 1: Verify Spark environment

print("Healthcare Bronze Ingestion Started")

print("Spark Version:", spark.version)

test_data = [
    (1, "Patient_A", "FL"),
    (2, "Patient_B", "TX"),
    (3, "Patient_C", "CA")
]

df_test = spark.createDataFrame(
    test_data,
    ["patient_id", "patient_name", "state"]
)

display(df_test)


In [0]:

# Step 2: Check ADLS Gen2 access

storage_account = "sthealthcarede2026"
container = "bronze"

file_path = (
    f"abfss://{container}@{storage_account}"
    ".dfs.core.windows.net/patients/patients.csv"
)

print("Checking file:", file_path)

try:
    files = dbutils.fs.ls(
        f"abfss://{container}@{storage_account}.dfs.core.windows.net/patients/"
    )

    for file in files:
        print(file.name, file.size)

except Exception as e:
    print("Storage access check failed.")
    print(str(e)[:1500])


In [0]:
bronze_path = (
    "abfss://bronze@sthealthcarede2026.dfs.core.windows.net/"
    "patients/patients.csv"
)

try:
    df_patients = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(bronze_path)
    )

    df_patients.show()
    print("Patient count:", df_patients.count())

except Exception as e:
    print("Storage read failed:")
    print(str(e)[:1500])

In [0]:

bronze_path = (
    "abfss://bronze@sthealthcarede2026.dfs.core.windows.net/"
    "patients/patients.csv"
)

try:
    df_patients = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(bronze_path)
    )

    df_patients.show(10, truncate=False)
    print("Total patient records:", df_patients.count())

except Exception as e:
    print("Storage read failed:")
    print(str(e)[:2000])


In [0]:
bronze_path = (
    "abfss://bronze@sthealthcarede2026.dfs.core.windows.net/"
    "patients/patients.csv"
)

df_patients = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(bronze_path)
)

df_patients.show(10, truncate=False)
print("Total patient records:", df_patients.count())

In [0]:
df_patients = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(bronze_path)
)

print("Total patient records:", df_patients.count())

In [0]:
bronze_path = (
    "abfss://bronze@sthealthcarede2026.dfs.core.windows.net/"
    "patients/patients.csv"
)

df_patients = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(bronze_path)
)

df_patients.show(10, truncate=False)
print("Total patient records:", df_patients.count())

In [0]:
from pyspark.sql.functions import current_timestamp, lit

batch_id = "BATCH_001"

df_bronze = (
    df_patients
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit(bronze_path))
    .withColumn("batch_id", lit(batch_id))
)

df_bronze.show(truncate = False)
df_bronze.printSchema()

In [0]:
print("Bronze record Count: ", df_bronze.count())
print("Bronze column count: ", len(df_bronze.columns))

assert df_bronze.count() == 5
assert len(df_bronze.columns) == 9

print("Bronze validation passed")

In [0]:
# check the current catalog and schema
spark.sql("SELECT current_catalog(), current_schema()").show()

# list avilable catalogs
spark.sql("SHOW CATALOGS").show()


In [0]:
# use the current workspace catalog
catalog_name = spark.sql(
    "SELECT current_catalog()"
).first()[0]

print("Catalog: ", catalog_name)

# Create Bronze schema if it doesn't exist
spark.sql(f"""
          CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.bronze
          """)

# Verify schema
spark.sql(f"""
          SHOW SCHEMAS IN `{catalog_name}`
          """).show()